# ScholarScan — Similarity Scoring & Grading

Compares a student's answer against a model (reference) answer and produces a score + grade.

## V2 Scoring Formula (5 components)

| Component | Weight | Description |
|-----------|--------|-------------|
| **SBERT** | 35% | Sentence-BERT semantic embedding cosine similarity |
| **Sentence** | 25% | Cross-sentence alignment (best-match recall + precision) |
| **Concept** | 25% | Hybrid keyword/phrase coverage (fuzzy + semantic) |
| **TF-IDF** | 10% | Lexical bigram cosine similarity |
| **NLI** | 5% | Neural entailment (optional, disabled by default) |

Weights redistribute proportionally if a component is unavailable (e.g. NLI disabled).

## Grade Bands
| Score | Grade | Marks fraction |
|-------|-------|----------------|
| ≥ 0.80 | A | 100% |
| 0.65–0.80 | B | 75% |
| 0.50–0.65 | C | 55% |
| 0.35–0.50 | D | 35% |
| < 0.35 | F | 10% |

In [ ]:
!pip install -q sentence-transformers scikit-learn numpy rapidfuzz spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
import re, logging, time
from dataclasses import dataclass, field
from functools import lru_cache
from typing import Literal, NamedTuple

import numpy as np
from rapidfuzz import fuzz as _rfuzz
from rapidfuzz import process as _rfuzz_process
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util

logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s')
print('✓ Imports OK')

## NLP Preprocessing (inline from nlp.py)

Needed before scoring — same pipeline as `02_nlp_preprocessing.ipynb`.

In [ ]:
# Source: backend/core/nlp.py (minimal subset needed for scoring)
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

_TOKEN_PATTERN = re.compile(r"\b\w+\b")
_OCR_TABLE = str.maketrans({"‘":"'","’":"'","“":'"',"”":'"',"–":"-","—":"-"," ":" "})

def _sanitize(t):
    t = t.translate(_OCR_TABLE).lower()
    t = t.encode("ascii", errors="ignore").decode()
    t = re.sub(r"[^a-z0-9\s\.,;:!?\-/'()%]", " ", t)
    t = re.sub(r"([.,;:!?])\1+", r"\1", t)
    return re.sub(r"\s+", " ", t).strip()

@lru_cache(maxsize=1)
def _get_spacy_nlp():
    import spacy
    return spacy.load("en_core_web_sm")

def preprocess_for_sbert(text):
    return _sanitize(text)

def preprocess_for_tfidf(text):
    sanitized = _sanitize(text)
    tokens = re.findall(r"\b\w+\b", sanitized)
    stop = frozenset(ENGLISH_STOP_WORDS)
    filtered = [t for t in tokens if t.isalpha() and t not in stop]
    if not filtered:
        return sanitized
    try:
        nlp = _get_spacy_nlp()
        doc = nlp(" ".join(filtered))
        lemmas = [tok.lemma_ for tok in doc if tok.lemma_.strip()]
        return " ".join(lemmas)
    except Exception:
        return " ".join(filtered)

def split_sentences(text):
    if not text.strip():
        return []
    try:
        nlp = _get_spacy_nlp()
        return [s.text.strip() for s in nlp(text).sents if len(s.text.split()) >= 3]
    except Exception:
        return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.split()) >= 3]

print('✓ Preprocessing helpers defined')

## Component A: TF-IDF Cosine Similarity

Lexical overlap using TF-IDF weighted bigrams. Fast, no model loading needed.  
Best at catching exact / near-exact wording.

In [ ]:
# Source: backend/core/similarity.py — TFIDFSimilarity

class TFIDFSimilarity:
    """Cosine similarity between two texts using TF-IDF vectors."""

    def __init__(self) -> None:
        self._vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)

    def compute(self, student_text: str, model_text: str) -> float:
        if not student_text.strip() or not model_text.strip():
            return 0.0
        tfidf_matrix = self._vectorizer.fit_transform([student_text, model_text])
        score = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0]
        return round(float(np.clip(score, 0.0, 1.0)), 4)

    def get_vectorizer(self):
        return self._vectorizer


# Demo
tfidf = TFIDFSimilarity()

pairs = [
    ("machine learning uses data",
     "machine learning uses data",
     "Identical"),
    ("machine learning trains models on data to make predictions",
     "machine learning uses data for training predictive models",
     "Paraphrase"),
    ("the weather is sunny today",
     "machine learning uses labelled training data",
     "Unrelated"),
]

print("=== TF-IDF Similarity ===")
print(f"{'Pair':<12} {'Score':>7}  {'Student':<45} {'Model'}")
print("-" * 100)
for label, (s, m) in [(p[2], (p[0], p[1])) for p in pairs]:
    score = tfidf.compute(preprocess_for_tfidf(s), preprocess_for_tfidf(m))
    print(f"{label:<12} {score:>7.4f}  {s[:43]:<45} {m[:45]}")

## Component B: SBERT Semantic Similarity

Sentence-BERT (`all-MiniLM-L6-v2`) encodes full sentences into 384-dim vectors.  
Cosine similarity catches semantic equivalence even with different wording.

In [ ]:
# Source: backend/core/similarity.py — SBERTSimilarity

class SBERTSimilarity:
    """Semantic similarity using Sentence-BERT embeddings."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2") -> None:
        self._model_name = model_name
        self._model = None
        print(f"Loading SBERT model: {model_name} ...")
        try:
            self._model = SentenceTransformer(model_name)
            print("✓ SBERT model ready")
        except Exception as e:
            print(f"⚠ SBERT unavailable: {e}")

    @property
    def is_available(self) -> bool:
        return self._model is not None

    def embed(self, text: str):
        if self._model is None:
            raise RuntimeError("SBERT not available")
        return self._model.encode(text, convert_to_tensor=True)

    def compute(self, student_text: str, model_text: str) -> float:
        if not student_text.strip() or not model_text.strip() or self._model is None:
            return 0.0
        emb_s = self.embed(student_text)
        emb_m = self.embed(model_text)
        return round(float(np.clip(util.cos_sim(emb_s, emb_m).item(), 0.0, 1.0)), 4)

    def batch_compute(self, student_answers: list, model_answer: str) -> list:
        if not model_answer.strip() or self._model is None:
            return [0.0] * len(student_answers)
        emb_m = self.embed(model_answer)
        emb_s = self._model.encode(student_answers, convert_to_tensor=True)
        scores = util.cos_sim(emb_s, emb_m).squeeze().tolist()
        if isinstance(scores, float):
            scores = [scores]
        return [round(float(np.clip(s, 0.0, 1.0)), 4) for s in scores]


sbert = SBERTSimilarity()

if sbert.is_available:
    pairs = [
        ("machine learning trains models on data to make predictions",
         "machine learning uses data for training predictive models",
         "Paraphrase"),
        ("supervised learning requires labelled training examples",
         "in supervised ML the training set contains ground truth labels",
         "Synonym/rephrase"),
        ("the weather is sunny today",
         "machine learning uses labelled training data",
         "Unrelated"),
    ]
    print("\n=== SBERT Similarity ===")
    print(f"{'Pair':<18} {'TF-IDF':>8} {'SBERT':>8}")
    print("-" * 50)
    for label, s, m in [(p[2], p[0], p[1]) for p in pairs]:
        ts = tfidf.compute(preprocess_for_tfidf(s), preprocess_for_tfidf(m))
        ss = sbert.compute(preprocess_for_sbert(s), preprocess_for_sbert(m))
        print(f"{label:<18} {ts:>8.4f} {ss:>8.4f}")
    print("\nNote: SBERT scores paraphrases much higher than TF-IDF does.")

## Component C: Hybrid Keyword / Concept Coverage

Extracts **noun phrases** from model answer → checks each phrase against student answer using:
1. **Fuzzy match** (RapidFuzz `token_set_ratio ≥ 85`) — catches OCR noise, minor spelling
2. **Semantic match** (SBERT cosine ≥ 0.70) — catches synonyms & paraphrases

Phrases are weighted by TF-IDF importance.

In [ ]:
# Source: backend/core/similarity.py — extract_phrases, hybrid_match, keyword_score

_FUZZY_KW_THRESHOLD = 85
_SBERT_KW_THRESHOLD = 0.70

@lru_cache(maxsize=1)
def _get_nlp_model():
    try:
        import spacy; return spacy.load("en_core_web_sm")
    except OSError:
        return None


def extract_phrases(text: str, nlp) -> list:
    """Extract noun chunks; fall back to non-stop tokens."""
    if nlp is None:
        return list(dict.fromkeys(t.lower() for t in text.split() if len(t) > 1))
    doc = nlp(text)
    phrases = [c.lemma_.lower().strip() for c in doc.noun_chunks if len(c.text.strip()) > 1]
    phrases = list(dict.fromkeys(phrases))
    if not phrases:
        phrases = list(dict.fromkeys(
            tok.lemma_.lower() for tok in doc
            if not tok.is_stop and not tok.is_punct and len(tok.text) > 1
        ))
    return phrases


def _compute_phrase_weights(phrases, vectorizer) -> dict:
    if not phrases: return {}
    try:
        vectorizer.get_feature_names_out()
    except Exception:
        return {p: 1.0 for p in phrases}
    doc = " ".join(phrases)
    vec_scores = vectorizer.transform([doc]).toarray()[0]
    fw = dict(zip(vectorizer.get_feature_names_out(), vec_scores))
    raw = {}
    for phrase in phrases:
        tokens = phrase.split()
        bigrams = [f"{tokens[i]} {tokens[i+1]}" for i in range(len(tokens)-1)]
        scores = [fw.get(c, 0.0) for c in tokens + bigrams]
        raw[phrase] = max(scores) if scores else 0.0
    max_w = max(raw.values()) or 1.0
    return {p: round(0.3 + 0.7 * (raw[p] / max_w), 4) for p in phrases}


def hybrid_match(model_phrase, student_phrases, sbert_row=None) -> bool:
    if not student_phrases: return False
    result = _rfuzz_process.extractOne(
        model_phrase, student_phrases, scorer=_rfuzz.token_set_ratio,
        score_cutoff=_FUZZY_KW_THRESHOLD,
    )
    if result is not None: return True
    if sbert_row is not None and float(sbert_row.max()) >= _SBERT_KW_THRESHOLD: return True
    return False


def keyword_score(model_phrases, student_phrases, weights, sbert_model=None):
    if not model_phrases: return 0.0, [], []
    sim_matrix = None
    if sbert_model is not None and sbert_model.is_available and student_phrases:
        try:
            all_embs = sbert_model._model.encode(model_phrases + student_phrases, convert_to_tensor=True)
            m_embs = all_embs[:len(model_phrases)]
            s_embs = all_embs[len(model_phrases):]
            sim_matrix = util.cos_sim(m_embs, s_embs)
        except Exception:
            pass
    matched, missing = [], []
    total_weight = sum(weights.values()) or 1.0
    matched_weight = 0.0
    for i, phrase in enumerate(model_phrases):
        w = weights.get(phrase, 1.0)
        sbert_row = sim_matrix[i] if sim_matrix is not None else None
        if hybrid_match(phrase, student_phrases, sbert_row=sbert_row):
            matched.append(phrase); matched_weight += w
        else:
            missing.append(phrase)
    return round(matched_weight / total_weight, 4), matched, missing


print('✓ Keyword/concept scoring functions defined')

In [ ]:
# Demo: concept coverage
nlp = _get_nlp_model()
vectorizer = tfidf.get_vectorizer()

MODEL_ANSWER = (
    "Supervised learning is a machine learning paradigm where the model is trained "
    "on labelled examples. The training data consists of input-output pairs. "
    "Common algorithms include decision trees, support vector machines, and neural networks. "
    "The model learns a mapping function and generalises to unseen data."
)

STUDENT_ANSWERS = {
    "Excellent (A)": (
        "Supervised learning trains on labelled data with input-output pairs. "
        "It uses algorithms like decision trees, SVMs, and neural networks. "
        "The trained model generalises to new unseen examples."
    ),
    "Partial (C)": (
        "Supervised learning uses labelled data to train models. "
        "Neural networks are one type of algorithm."
    ),
    "Weak (D)": (
        "Machine learning is AI. It learns things automatically."
    ),
}

# Fit vectorizer on model answer
model_tfidf = preprocess_for_tfidf(MODEL_ANSWER)
vectorizer.fit([model_tfidf])
model_phrases = extract_phrases(MODEL_ANSWER, nlp)
weights = _compute_phrase_weights(model_phrases, vectorizer)

print(f"Model phrases ({len(model_phrases)}): {model_phrases}")
print()

for label, answer in STUDENT_ANSWERS.items():
    student_phrases = extract_phrases(answer, nlp)
    ratio, matched, missing = keyword_score(model_phrases, student_phrases, weights, sbert)
    print(f"--- {label} ---")
    print(f"  Coverage : {ratio:.2%}")
    print(f"  Matched  : {matched}")
    print(f"  Missing  : {missing}")
    print()

## Component D: Sentence-Level Similarity

Splits both texts into sentences, computes all-pairs SBERT cosine similarity matrix.  
Score = `0.6 × recall (model coverage) + 0.4 × precision (student relevance)`.

In [ ]:
# Source: backend/core/similarity.py — SentenceSimilarityScorer
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

_MAX_SENTENCES = 128

@dataclass
class SentenceSimilarityResult:
    score: float
    matrix: np.ndarray | None = None
    best_matches: list = field(default_factory=list)
    coverage_precision: float = 0.0


class SentenceSimilarityScorer:
    """Cross-sentence alignment using SBERT."""

    def __init__(self, sbert_model: SBERTSimilarity) -> None:
        self._sbert = sbert_model

    def score(self, student_text: str, model_text: str) -> SentenceSimilarityResult:
        if not self._sbert.is_available:
            return SentenceSimilarityResult(score=0.0)

        model_sents = split_sentences(model_text)[:_MAX_SENTENCES]
        student_sents = split_sentences(student_text)[:_MAX_SENTENCES]

        if not model_sents or not student_sents:
            return SentenceSimilarityResult(score=0.0)

        all_embs = self._sbert._model.encode(
            model_sents + student_sents, convert_to_tensor=True
        )
        m_embs = all_embs[:len(model_sents)]
        s_embs = all_embs[len(model_sents):]
        sim_matrix = util.cos_sim(m_embs, s_embs).cpu().numpy()

        # Recall: how well does student cover each model sentence?
        recall = float(np.mean(np.max(sim_matrix, axis=1)))
        # Precision: how relevant is each student sentence to model?
        precision = float(np.mean(np.max(sim_matrix, axis=0)))
        combined = round(0.6 * recall + 0.4 * precision, 4)

        best_matches = []
        for i, ms in enumerate(model_sents):
            best_j = int(np.argmax(sim_matrix[i]))
            best_matches.append((ms, student_sents[best_j], float(sim_matrix[i, best_j])))

        return SentenceSimilarityResult(
            score=combined,
            matrix=sim_matrix,
            best_matches=best_matches,
            coverage_precision=precision,
        )


print('✓ SentenceSimilarityScorer defined')

In [ ]:
# Demo: sentence similarity with heatmap
if sbert.is_available:
    sent_scorer = SentenceSimilarityScorer(sbert)
    excellent_answer = STUDENT_ANSWERS["Excellent (A)"]
    result = sent_scorer.score(excellent_answer, MODEL_ANSWER)

    print(f"Sentence similarity score: {result.score:.4f}")
    print(f"  Recall   : {float(np.mean(np.max(result.matrix, axis=1))):.4f}")
    print(f"  Precision: {result.coverage_precision:.4f}")
    print()
    print("Best sentence alignments:")
    for ms, ss, sim in result.best_matches[:4]:
        print(f"  [{sim:.3f}] MODEL  : {ms[:60]}")
        print(f"           STUDENT: {ss[:60]}")
        print()

    # Heatmap
    m_sents = split_sentences(MODEL_ANSWER)[:6]
    s_sents = split_sentences(excellent_answer)[:6]
    matrix = result.matrix[:len(m_sents), :len(s_sents)]

    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(matrix, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(s_sents)))
    ax.set_yticks(range(len(m_sents)))
    ax.set_xticklabels([f"S{i+1}: {s[:25]}..." for i, s in enumerate(s_sents)],
                       rotation=30, ha='right', fontsize=8)
    ax.set_yticklabels([f"M{i+1}: {s[:30]}..." for i, s in enumerate(m_sents)], fontsize=8)
    for i in range(len(m_sents)):
        for j in range(len(s_sents)):
            ax.text(j, i, f"{matrix[i,j]:.2f}", ha='center', va='center',
                   fontsize=9, color='black' if matrix[i,j] < 0.6 else 'white')
    plt.colorbar(im, ax=ax, label='Cosine Similarity')
    ax.set_xlabel('Student sentences')
    ax.set_ylabel('Model sentences')
    ax.set_title('Sentence-Level Similarity Matrix', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Combined Score: V2 Formula

Assembles all components with weight redistribution when a component is unavailable.

In [ ]:
# Source: backend/core/similarity.py — _redistribute_weights, compute_similarity (simplified)

def _redistribute_weights(base_weights: dict, available: dict) -> dict:
    """Redistribute unavailable component weights proportionally."""
    active = {k: v for k, v in base_weights.items() if available.get(k, True)}
    if not active: return base_weights
    total = sum(active.values())
    if total == 0: return active
    return {k: round(v / total, 4) for k, v in active.items()}


def compute_similarity_v2(
    student_text: str,
    model_text: str,
    sbert_model: SBERTSimilarity,
    *,
    weight_sbert: float = 0.35,
    weight_sentence: float = 0.25,
    weight_concept: float = 0.25,
    weight_tfidf: float = 0.10,
    weight_nli: float = 0.05,
) -> dict:
    """5-component V2 scoring pipeline."""
    # Preprocess
    student_tfidf = preprocess_for_tfidf(student_text)
    model_tfidf   = preprocess_for_tfidf(model_text)
    student_sbert = preprocess_for_sbert(student_text)
    model_sbert   = preprocess_for_sbert(model_text)

    tfidf_scorer = TFIDFSimilarity()

    # A: TF-IDF
    tfidf_score = tfidf_scorer.compute(student_tfidf, model_tfidf)

    # B: SBERT
    sbert_score = sbert_model.compute(student_sbert, model_sbert) if sbert_model.is_available else 0.0

    # C: Concept coverage
    nlp = _get_nlp_model()
    tfidf_scorer._vectorizer.fit([model_tfidf])
    model_phrases   = extract_phrases(model_text, nlp)
    student_phrases = extract_phrases(student_text, nlp)
    weights         = _compute_phrase_weights(model_phrases, tfidf_scorer._vectorizer)
    concept_ratio, matched, missing = keyword_score(model_phrases, student_phrases, weights, sbert_model)

    # D: Sentence similarity
    sentence_score = 0.0
    sent_result    = None
    if sbert_model.is_available:
        sent_scorer = SentenceSimilarityScorer(sbert_model)
        sent_result = sent_scorer.score(student_sbert, model_sbert)
        sentence_score = sent_result.score

    # Weights
    available = {
        "sbert":    sbert_model.is_available,
        "sentence": sbert_model.is_available,
        "concept":  True,
        "tfidf":    True,
        "nli":      False,  # NLI disabled by default (requires heavy transformer model)
    }
    base_weights = {
        "sbert": weight_sbert, "sentence": weight_sentence,
        "concept": weight_concept, "tfidf": weight_tfidf, "nli": weight_nli,
    }
    eff = _redistribute_weights(base_weights, available)

    combined = (
        eff.get("sbert",    0) * sbert_score
        + eff.get("sentence", 0) * sentence_score
        + eff.get("concept",  0) * concept_ratio
        + eff.get("tfidf",    0) * tfidf_score
    )
    combined = round(float(np.clip(combined, 0.0, 1.0)), 4)

    return {
        "tfidf_score":       tfidf_score,
        "sbert_score":       sbert_score,
        "sentence_similarity": sentence_score,
        "concept_coverage":  concept_ratio,
        "combined_score":    combined,
        "keyword_overlap":   concept_ratio,
        "matched_keywords":  matched,
        "missing_keywords":  missing,
        "effective_weights": eff,
    }


print('✓ compute_similarity_v2 defined')

## Grade Bands & Marks

Maps combined score → letter grade → marks fraction.

In [ ]:
# Source: backend/core/scoring.py

class _GradeBand(NamedTuple):
    min_score: float
    max_score: float
    letter: str
    marks_fraction: float
    feedback_key: str


_GRADE_BANDS = (
    _GradeBand(0.80, 1.01, "A", 1.00, "excellent"),
    _GradeBand(0.65, 0.80, "B", 0.75, "good"),
    _GradeBand(0.50, 0.65, "C", 0.55, "partial"),
    _GradeBand(0.35, 0.50, "D", 0.35, "weak"),
    _GradeBand(0.00, 0.35, "F", 0.10, "insufficient"),
)

_FEEDBACK_TEMPLATES = {
    "excellent":    "Excellent answer! Covered all key concepts clearly.",
    "good":         "Good answer. Most key concepts covered; minor gaps present.",
    "partial":      "Partial answer. Core ideas present but several concepts missing.",
    "weak":         "Weak answer. Only a few relevant points addressed.",
    "insufficient": "Insufficient. Answer does not adequately address the question.",
}


def _assign_grade(score: float) -> _GradeBand:
    for band in _GRADE_BANDS:
        if band.min_score <= score < band.max_score:
            return band
    return _GRADE_BANDS[-1]


@dataclass(frozen=True)
class AssessmentResult:
    question_id: str
    student_id: str
    tfidf_score: float
    sbert_score: float
    sentence_similarity: float
    concept_coverage: float
    combined_score: float
    keyword_overlap: float
    matched_keywords: tuple
    missing_keywords: tuple
    marks: float
    max_marks: int
    grade: str
    feedback: str

    def summary(self) -> str:
        bar = lambda v: "█" * int(v * 20) + "░" * (20 - int(v * 20))
        return (
            f"\n{'═'*60}\n"
            f"  Question : {self.question_id}  |  Student: {self.student_id}\n"
            f"{'─'*60}\n"
            f"  TF-IDF    [{bar(self.tfidf_score)}] {self.tfidf_score:.4f}\n"
            f"  SBERT     [{bar(self.sbert_score)}] {self.sbert_score:.4f}\n"
            f"  Sentence  [{bar(self.sentence_similarity)}] {self.sentence_similarity:.4f}\n"
            f"  Concepts  [{bar(self.concept_coverage)}] {self.concept_coverage:.4f}\n"
            f"  COMBINED  [{bar(self.combined_score)}] {self.combined_score:.4f}\n"
            f"{'─'*60}\n"
            f"  MARKS   : {self.marks} / {self.max_marks}   [{self.grade}]\n"
            f"  FEEDBACK: {self.feedback}\n"
            f"  Matched : {', '.join(self.matched_keywords[:5])}\n"
            f"  Missing : {', '.join(self.missing_keywords[:5])}\n"
            f"{'═'*60}\n"
        )


def score_answer(
    question_id: str,
    student_id: str,
    student_text: str,
    model_text: str,
    sbert_model: SBERTSimilarity,
    max_marks: int = 10,
) -> AssessmentResult:
    sim = compute_similarity_v2(student_text, model_text, sbert_model)
    score = sim["combined_score"]
    band = _assign_grade(score)
    return AssessmentResult(
        question_id=question_id,
        student_id=student_id,
        tfidf_score=sim["tfidf_score"],
        sbert_score=sim["sbert_score"],
        sentence_similarity=sim["sentence_similarity"],
        concept_coverage=sim["concept_coverage"],
        combined_score=score,
        keyword_overlap=sim["keyword_overlap"],
        matched_keywords=tuple(sim["matched_keywords"]),
        missing_keywords=tuple(sim["missing_keywords"]),
        marks=round(band.marks_fraction * max_marks, 1),
        max_marks=max_marks,
        grade=band.letter,
        feedback=_FEEDBACK_TEMPLATES[band.feedback_key],
    )


print('✓ AssessmentResult + score_answer defined')

## End-to-End Example

Grade 3 student answers (Excellent / Partial / Weak) against the same model answer.

In [ ]:
QUESTION = "Explain supervised learning and name common algorithms."

MODEL_ANSWER = """
Supervised learning is a machine learning paradigm where the model is trained on labelled
examples. The training data consists of input-output pairs. The model learns a mapping
function from inputs to outputs and generalises to unseen data. Common algorithms include
decision trees, support vector machines (SVM), and neural networks.
""".strip()

STUDENT_CASES = [
    ("student_001", "Excellent", """
        Supervised learning trains on labelled input-output pairs to learn a mapping function.
        It generalises learned patterns to new unseen data. Key algorithms are decision trees,
        support vector machines, and neural networks.
    """),
    ("student_002", "Partial", """
        Supervised learning uses labelled data to train machine learning models. Neural networks
        and decision trees are types of supervised algorithms.
    """),
    ("student_003", "Weak", """
        Machine learning is part of AI. Computers can learn automatically from data.
        There are many different algorithms available.
    """),
]

results = []
for sid, label, answer in STUDENT_CASES:
    r = score_answer(
        question_id="Q1",
        student_id=f"{sid} ({label})",
        student_text=answer.strip(),
        model_text=MODEL_ANSWER,
        sbert_model=sbert,
        max_marks=10,
    )
    results.append(r)
    print(r.summary())

In [ ]:
# Score comparison bar chart
import matplotlib.pyplot as plt
import numpy as np

categories = ["TF-IDF", "SBERT", "Sentence", "Concepts", "Combined"]
colors = ['#3498db', '#27ae60', '#f39c12', '#e74c3c', '#8e44ad']

fig, axes = plt.subplots(1, len(results), figsize=(15, 5), sharey=True)
fig.patch.set_facecolor('#1a1a2e')

for ax, r in zip(axes, results):
    values = [
        r.tfidf_score, r.sbert_score,
        r.sentence_similarity, r.concept_coverage, r.combined_score,
    ]
    bars = ax.bar(categories, values, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_ylim(0, 1.05)
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='white')
    ax.set_xticklabels(categories, rotation=30, ha='right', color='white', fontsize=9)
    ax.axhline(0.80, color='#2ecc71', linestyle='--', linewidth=1, alpha=0.7)
    ax.axhline(0.50, color='#e67e22', linestyle='--', linewidth=1, alpha=0.7)
    ax.text(4, 0.81, 'A', color='#2ecc71', fontsize=8)
    ax.text(4, 0.51, 'C', color='#e67e22', fontsize=8)
    title_color = {'A': '#2ecc71', 'B': '#27ae60', 'C': '#f39c12', 'D': '#e67e22', 'F': '#e74c3c'}[r.grade]
    ax.set_title(f"{r.student_id}\nGrade {r.grade}  {r.marks}/{r.max_marks}",
                 color=title_color, fontsize=10, fontweight='bold')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.2f}',
                ha='center', va='bottom', fontsize=8, color='white')

axes[0].set_ylabel('Score (0–1)', color='white')
plt.suptitle('ScholarScan — V2 Scoring Component Breakdown', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Interactive: Test Your Own Answers

In [ ]:
# ── Edit these ──
MY_QUESTION = "What is reinforcement learning?"

MY_MODEL_ANSWER = """
Reinforcement learning is a type of machine learning where an agent learns to make decisions
by interacting with an environment. The agent receives reward signals for actions and learns
an optimal policy that maximises cumulative reward. Key algorithms include Q-learning,
policy gradients, and actor-critic methods.
""".strip()

MY_STUDENT_ANSWER = """
Reinforcement learning trains an agent to take actions in an environment to maximise rewards.
It learns by trial and error using Q-learning and policy gradient algorithms.
""".strip()
# ── ──

result = score_answer(
    question_id="Custom Q",
    student_id="your_student",
    student_text=MY_STUDENT_ANSWER,
    model_text=MY_MODEL_ANSWER,
    sbert_model=sbert,
    max_marks=10,
)
print(result.summary())